RECOMENDATION APP

In [45]:
import pandas as pd
import numpy as np
from data_load import df

In [46]:
# Filtrujeme pouze explicitní hodnocení
df = df[df["Book-Rating"] > 0]
print(f"Explicit ratings: {len(df):,} ({len(df)/len(df)*100:.1f}%)")

Explicit ratings: 383,842 (100.0%)


1. IDENTIFIKACE FANBASE

In [47]:
INPUT_BOOK = "The Lord of the Rings"
FAN_THRESHOLD = 8

# Všechna ISBN odpovídající vstupní knize
input_book_isbns = df[
    df["Book-Title"].str.contains(INPUT_BOOK, na=False, case=False)
]["ISBN"].unique()

print(f"Nalezeno {len(input_book_isbns)} ISBN pro '{INPUT_BOOK}'")

# uživatele s ratingem >= FAN_THRESHOLD pro alespoň jedno z těchto ISBN
fans = df[
    (df["ISBN"].isin(input_book_isbns)) & 
    (df["Book-Rating"] >= FAN_THRESHOLD)
]["User-ID"].unique()

print(f"Nalezeno {len(fans)} fanoušků knihy '{INPUT_BOOK}'")

Nalezeno 86 ISBN pro 'The Lord of the Rings'
Nalezeno 442 fanoušků knihy 'The Lord of the Rings'


2. OTHER FANBASE BOOKS

In [48]:
# Co dalšího fanoušci hodnotili

fan_ratings = df[
    (df["User-ID"].isin(fans)) & 
    (~df["ISBN"].isin(input_book_isbns))
]

print(f"Počet hodnocení od fanoušků: {len(fan_ratings):,}")
print(f"Počet unikátních knih, které hodnotili: {fan_ratings['ISBN'].nunique():,}")

Počet hodnocení od fanoušků: 32,993
Počet unikátních knih, které hodnotili: 25,165


3. BOOK STATISTICS

In [49]:
# Knižní statistiky

book_stats = fan_ratings.groupby(["ISBN", "Book-Title", "Book-Author"]).agg(
    fan_count=("Book-Rating", "size"),
    avg_rating=("Book-Rating", "mean"),
).reset_index()

print(f"Počet unikátních knih: {len(book_stats):,}")
print(book_stats.head())

Počet unikátních knih: 25,165
         ISBN                                         Book-Title  \
0  0000913154  The Way Things Work: An Illustrated Encycloped...   
1  0001047973                                    Brave New World   
2  0001055607                             Cereus Blooms At Night   
3  0001845039                                The Moon of Gomrath   
4  0001944711                    Count Duckula: Vampire Vacation   

                     Book-Author  fan_count  avg_rating  
0  C. van Amerongen (translator)          1         8.0  
1                  Aldous Huxley          1         9.0  
2                   Shani Mootoo          1         8.0  
3                    Alan Garner          1        10.0  
4               Maureen Spurgeon          1         6.0  


4. NUMBER OF RATINGS THRESHOLD

In [50]:
# Filtr na knihy s dostatečným signálem

MIN_FANS = 1

#book_stats_filtered = book_stats[book_stats["fan_count"] >= MIN_FANS]


"Skóre kombinuje počet fanoušků a jejich hodnocení. Hodnocení přemapuju mocninnou křivkou (nízké hodnocení potlačím, vysoká zdůrazním), fanoušky odmocninou (první přírůstky mají velkou váhu, další menší). Tím se vyvažuje kvalita vs. popularita."

4. TOP 10 BOOKS

In [51]:
TOP_N = 10
K_RATING = 2   # křivka pro hodnocení - nízké potlačíme, vysoké necháme růst
K_FANS = 0.5  # křivka pro fanoušky - první přírůstky mají velkou váhu, další menší

book_stats = book_stats.copy()

# Přemapování hodnocení na škálu 1-10 s ohnutím křivky
book_stats["adjusted_rating"] = (
    1 + 9 * ((book_stats["avg_rating"] - 1) / 9) ** K_RATING
)

# Přemapování počtu fanoušků na škálu 1-10, relativně k maximu v aktuálním výsledku
max_fans = book_stats["fan_count"].max()
book_stats["adjusted_fans"] = (
    1 + 9 * (book_stats["fan_count"] / max_fans) ** K_FANS
)

# Finální skóre: rozsah 1-100
book_stats["score"] = (
    book_stats["adjusted_rating"] * book_stats["adjusted_fans"]
)

recommendations = book_stats.sort_values("score", ascending=False).head(TOP_N)

recommendations[["Book-Title", "Book-Author", "avg_rating", "fan_count"]]

,Book-Title,Book-Author,avg_rating,fan_count
8287,Harry Potter and the Order of the Phoenix (Boo...,J. K. Rowling,9.282051,39
13285,Harry Potter and the Sorcerer's Stone (Harry P...,J. K. Rowling,9.083333,36
8231,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,9.176471,34
8212,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,9.114286,35
8234,Harry Potter and the Goblet of Fire (Book 4),J. K. Rowling,9.500000,28
13284,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,9.307692,26
6834,The Da Vinci Code,Dan Brown,8.870968,31
8213,Harry Potter and the Chamber of Secrets (Book 2),J. K. Rowling,8.880000,25
8232,Harry Potter and the Prisoner of Azkaban (Book 3),J. K. Rowling,9.315789,19
9237,To Kill a Mockingbird,Harper Lee,9.150000,20
